## Manuel Ramallo, Lucía Pérez, Alexandre Lorenzo

# Implementación dunha arquitectura MLP modular


In [1]:
import torch
from torch import nn

class MLP(nn.Module):

    def __init__(self, input_neurons, output_neurons, hidden_layers):
        
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        
        # Lista para almacenar capas
        self.layers = nn.ModuleList()
        
        # Engadimos a primeira capa (Entrada -> Primeira capa oculta)
        # Se a lista de ocultas está vacía, conectamos entrada con saída directamente
        if len(hidden_layers) > 0:
            self.layers.append(nn.Linear(input_neurons, hidden_layers[0]))
            
            # Engadimos as capas ocultas intermedias
            # Percorremos dende a primeira oculta ata a penúltima
            for i in range(len(hidden_layers) - 1):
                self.layers.append(nn.Linear(hidden_layers[i], hidden_layers[i+1]))
            
            # Engadimos a última capa (Última oculta -> Saída)
            self.layers.append(nn.Linear(hidden_layers[-1], output_neurons))
        else:
            # Caso especial: Se non hai capas ocultas
            self.layers.append(nn.Linear(input_neurons, output_neurons))

        # Definimos as funcións de activación
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)


    def forward(self, x):
        
        x = self.flatten(x)
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = self.relu(x)
        
        return x

In [8]:
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import confusion_matrix
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, random_split
from torch.utils.data import DataLoader
import numpy as np
import random
import copy 
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

torch.manual_seed(1)
random.seed(1)
np.random.seed(1)


def plot_data_dist(training_labels, test_labels, labels_map):
    c_train = Counter(training_labels)
    class_freq_train = dict(c_train)

    c_test = Counter(test_labels)
    class_freq_test = dict(c_test)

    plt.figure(figsize=(10, 3))
    width = 0.4 
    train_keys = list(class_freq_train.keys())
    test_keys = [key + width for key in class_freq_test.keys()]

    plt.bar(train_keys, class_freq_train.values(), width=width, color='b', align='center', label='Adestramento')
    plt.bar(test_keys, class_freq_test.values(), width=width, color='r', align='center', label='Test')

    plt.xticks([key + width / 2 for key in class_freq_train.keys()], labels_map.values(), rotation='vertical')
    plt.ylabel('Frecuencia')
    plt.xlabel('Categoría')
    plt.legend()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, labels_map):
    conf_matrix = confusion_matrix(y_true, y_pred)
    conf_matrix_normalized = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 10))
    plt.imshow(conf_matrix_normalized, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion matrix')
    plt.colorbar()
    tick_marks = np.arange(10)
    plt.xticks(tick_marks, labels_map.values(), rotation=45)
    plt.yticks(tick_marks, labels_map.values())

    thresh = conf_matrix.max() / 2.
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            plt.text(j, i, f"{conf_matrix[i, j]}", 
                    horizontalalignment="center",
                    color="white" if conf_matrix[i, j] > thresh else "black")

    plt.ylabel('Clase real')
    plt.xlabel('Clase predicida')
    plt.show()

def get_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [9]:
def train(dataloader, model, loss_fn, optimizer):
    # Obtemos o número total de lotes
    num_images = len(dataloader.dataset)
    # Poñemos o modelo en modo de adestramento
    model.train()
    # Iteramos sobre os lotes
    n_batch = 0
    for batch_imgs, batch_labels in dataloader:
        # Obtemos as predicións para o lote actual
        batch_predicted_probabilities = model(batch_imgs)
        # Calculamos a perda
        loss = loss_fn(batch_predicted_probabilities, batch_labels)
        # Poñemos a cero os gradientes dos parámetros do modelo
        optimizer.zero_grad()
        # Realizamos a retropropagación para calcular os gradientes de cada parámetro
        loss.backward()
        # Actualizamos os parámetros do modelo utilizando os gradientes calculados
        optimizer.step()

        # Cada 100 lotes, imprimimos a perda e o progreso
        if n_batch % 100 == 0:
            loss, imgs_processed = loss.item(), n_batch * len(batch_imgs)
            print(f"perda: {loss:>7f}  [{imgs_processed:>5d}/{num_images:>5d}]")

        n_batch += 1

    return loss

def test(dataloader, model):
    # Obtemos o número total de imaxes
    num_images = len(dataloader.dataset)
    # Poñemos o modelo en modo de avaliación (sen calcular gradientes)
    model.eval()
    # Inicializamos as variables para gardar a perda e a precisión
    test_loss, correct = 0, 0
    # Inicializamos as listas para gardar as clases predicidas e as clases reais
    test_predicted_classes, test_true_classes = [], []
    with torch.no_grad():
        for batch_imgs, batch_labels in dataloader:
            # Obtemos as predicións para o lote actual
            batch_predicted_probabilities = model(batch_imgs)
            # Calculamos a perda e acumulamos o valor
            test_loss += loss_fn(batch_predicted_probabilities, batch_labels).item()
            # Calculamos as clases predicidas (aquelas con maior probabilidade)
            batch_predicted_classes = batch_predicted_probabilities.argmax(dim=1)
            # Obtemos o número de predicións correctas e acumulamos o valor
            correct += (batch_predicted_classes == batch_labels).sum().item()
            # Gardamos as clases predicidas e as clases reais. Precisamos convertelas a listas de Python coa función tolist()
            test_predicted_classes.extend(batch_predicted_classes.tolist())
            test_true_classes.extend(batch_labels.tolist())
    # Calculamos a perda media e o porcentaxe de predicións correctas (accuracy), dividindo entre o número total de imaxes 
    test_loss /= num_images
    correct /= num_images
    print(f"Perda media: {test_loss:>8f} Accuracy: {correct*100:>0.1f}%\n")

    return test_predicted_classes, test_true_classes

In [11]:
def test_loss(dataloader, model, loss_fn):
    # Obtemos o número total de imaxes
    num_images = len(dataloader.dataset)
    # Poñemos o modelo en modo de avaliación (sen calcular gradientes)
    model.eval()
    # Inicializamos as variables para gardar a perda e a precisión
    test_loss, correct = 0, 0
    # Inicializamos as listas para gardar as clases predicidas e as clases reais
    test_predicted_classes, test_true_classes = [], []
    with torch.no_grad():
        for batch_imgs, batch_labels in dataloader:
            # Obtemos as predicións para o lote actual
            batch_predicted_probabilities = model(batch_imgs)
            # Calculamos a perda e acumulamos o valor
            test_loss += loss_fn(batch_predicted_probabilities, batch_labels).item()
            # Calculamos as clases predicidas (aquelas con maior probabilidade)
            batch_predicted_classes = batch_predicted_probabilities.argmax(dim=1)
            # Obtemos o número de predicións correctas e acumulamos o valor
            correct += (batch_predicted_classes == batch_labels).sum().item()
            # Gardamos as clases predicidas e as clases reais. Precisamos convertelas a listas de Python coa función tolist()
            test_predicted_classes.extend(batch_predicted_classes.tolist())
            test_true_classes.extend(batch_labels.tolist())
    # Calculamos a perda media e o porcentaxe de predicións correctas (accuracy), dividindo entre o número total de imaxes 
    test_loss /= num_images
    correct /= num_images
    print(f"Perda media: {test_loss:>8f} Accuracy: {correct*100:>0.1f}%\n")

    return test_loss

## Optimización de MLP nun problema real

In [12]:
transform_imagenette = transforms.Compose([
    transforms.Resize((80, 80)),
    transforms.ToTensor(),
])

training_data = datasets.Imagenette(
    root="data",
    split="train",
    size="160px",
    download=True,
    transform=transform_imagenette
)

test_data = datasets.Imagenette(
    root="data",
    split="val",
    size="160px",
    download=True,
    transform=transform_imagenette
)

Probaremos as seguintes configuracións de neuronas en capas ocultas:
 - 3 capas: [1024, 512, 256]
 - 3 capas, pero con máis neuronas: [2048, 1024, 512]
 - Profunda, con repetición de número de neuronas en capas consecutivas: [1024, 1024, 512, 512, 256, 256]

In [13]:
mlps = [{"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[256]), "lr": 1e-3, "optimizer": "Adam"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512]), "lr": 1e-3, "optimizer": "Adam"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[1024, 512]), "lr": 1e-3, "optimizer": "Adam"}, 
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512, 256]), "lr": 1e-3, "optimizer": "Adam"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[1024, 512, 256]), "lr": 1e-3, "optimizer": "Adam"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512, 256]), "lr": 1e-2, "optimizer": "Adam"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512]), "lr": 1e-2, "optimizer": "Adam"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[256]), "lr": 1e-3, "optimizer": "SGD"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512]), "lr": 1e-3, "optimizer": "SGD"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[1024, 512]), "lr": 1e-3, "optimizer": "SGD"}, 
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512, 256]), "lr": 1e-3, "optimizer": "SGD"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[1024, 512, 256]), "lr": 1e-3, "optimizer": "SGD"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512, 256]), "lr": 1e-2, "optimizer": "SGD"},
        {"model": MLP(input_neurons=80*80*3, output_neurons=10, hidden_layers=[512]), "lr": 1e-2, "optimizer": "SGD"}
]

loss_fn = nn.CrossEntropyLoss()

In [16]:
train_size = int(0.85 * len(training_data))
val_size = len(training_data) - train_size

train_dataset, val_dataset = random_split(training_data, [train_size, val_size])

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

best_global_val_loss = float('inf')
best_global_model    = None
best_global_cfg      = None
results = []

for config in mlps:
    mlp = config["model"]
    learning_rate = config["lr"]
    optimizer_name = config["optimizer"]
    train_losses_mlp = []
    val_losses_mlp = []
    epochs = 1
    best_val_loss = float('inf')
    best_model = None
    patience = 3
    current_patience = 0

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(mlp.parameters(), lr=learning_rate)
    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(mlp.parameters(), lr=learning_rate)

    for t in range(epochs):
        print(f"Epoch {t+1}\n-------------------------------")
        train_loss = train(train_dataloader, mlp, loss_fn, optimizer)
        train_losses_mlp.append(train_loss.item())

        val_loss = test_loss(val_dataloader, mlp, loss_fn)
        val_losses_mlp.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = copy.deepcopy(mlp.state_dict())
            current_patience = 0
        else:
            current_patience += 1
            if current_patience >= patience:
                print("Early stopping")
                break

    mlp.load_state_dict(best_model)

    # Evaluar en test
    _, test_predicted, test_true = evaluate(test_dataloader, mlp, loss_fn)
    acc       = accuracy_score(test_true, test_predicted)
    precision = precision_score(test_true, test_predicted, average='macro')
    recall    = recall_score(test_true, test_predicted, average='macro')
    f1        = f1_score(test_true, test_predicted, average='macro')

    results.append({
        "hidden_layers": str(config["hidden_layers"]),
        "optimizer":     optimizer_name,
        "lr":            learning_rate,
        "params":        get_parameters(mlp),
        "val_loss":      best_val_loss,
        "accuracy":      acc,
        "precision":     precision,
        "recall":        recall,
        "f1":            f1,
        "train_losses":  train_losses_mlp,
        "val_losses":    val_losses_mlp,
    })

    if best_val_loss < best_global_val_loss:
        best_global_val_loss = best_val_loss
        best_global_model    = copy.deepcopy(mlp.state_dict())
        best_global_cfg      = config
        print(f"  ★ Novo mellor modelo global (val_loss={best_val_loss:.6f})")

# Tabla resumen
results_df = pd.DataFrame(results).drop(columns=["train_losses", "val_losses"])
print(results_df.to_string(index=False))

Epoch 1
-------------------------------
perda: 1.913117  [    0/ 8048]
perda: 1.905756  [ 3200/ 8048]
perda: 1.798565  [ 6400/ 8048]
Perda media: 0.060475 Accuracy: 36.0%



NameError: name 'evaluate' is not defined